In [14]:
from datasets.packaged_modules.pandas.pandas import Pandas
%load_ext autoreload
%autoreload 2
import os
import pandas as pd
from datasets import Dataset
from dotenv import load_dotenv
from DebtHunterSatdDetectorModel import DebtHunterSatdDetectorModel
from SimpleOutputLabelConverter import SimpleOutputLabelConverter
from constant import DEFAULT_DETECTION_CLASS

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
load_dotenv()
simple_output_label_converter = SimpleOutputLabelConverter({'yes', 'no'}, DEFAULT_DETECTION_CLASS)
CACHE_DIRECTORY = os.getenv('CACHE_DIRECTORY', '../cache')
BASE_DEBT_HUNTER_DIRECTORY = os.getenv(
    'BASE_DEBT_HUNTER_DIRECTORY',
    f'{CACHE_DIRECTORY}/baseline/debthunter',
)
os.makedirs(BASE_DEBT_HUNTER_DIRECTORY, exist_ok=True)
print('DebtHunter working directory:', BASE_DEBT_HUNTER_DIRECTORY)

DebtHunter working directory: ../cache/baseline/debthunter


# DebtHunter Baseline (RQ2)

Loops over the two RQ2 test sets and runs the patched DebtHunter JAR (which
exposes a `-csv` input mode via `parsing.CsvParsing`) on each:

| `dataset_name` | Dataset                         | File                              |
|----------------|---------------------------------|-----------------------------------|
| `unique`       | Deduplicated detection set      | `data/unique_detect_test.csv`     |
| `duplicate`    | Original detection set          | `data/duplicate_detect_test.csv`  |

For each dataset the notebook runs two variants:

* `pretrained-debthunter` — uses DebtHunter's released models
  (`DEBT_HUNTER_PRETRAINED_DIR`/{DHbinaryClassifier,DHmultiClassifier}.model).
* `trained-debthunter` — retrains via `-u second` on that dataset's train split,
  then scores the test split with the retrained models.

In [16]:
for dataset_name in ['unique', 'duplicate']:
    print(f'\n===== {dataset_name} =====')
    detect_train_df = pd.read_csv(f'../data/{dataset_name}_detect_train.csv')
    detect_test_df = pd.read_csv(f'../data/{dataset_name}_detect_test.csv')
    detect_train_dataset = Dataset.from_pandas(detect_train_df)
    detect_test_dataset = Dataset.from_pandas(detect_test_df)

    # Pretrained DebtHunter
    pretrained_detector = DebtHunterSatdDetectorModel(
        'detect',
        'pretrained-debthunter',
        simple_output_label_converter,
        BASE_DEBT_HUNTER_DIRECTORY,
    )
    pretrained_detector.fit(detect_train_dataset)
    pretrained_detector.predict(detect_test_dataset, dataset_name)

    # Retrained DebtHunter
    trained_detector = DebtHunterSatdDetectorModel(
        'detect',
        'trained-debthunter',
        simple_output_label_converter,
        BASE_DEBT_HUNTER_DIRECTORY,
        retrain=True,
    )
    trained_detector.fit(detect_train_dataset)
    trained_detector.predict(detect_test_dataset, dataset_name)


===== unique =====
pretrained-debthunter
Let's start!
You selected the first use case!
Loaded 6531 comment rows from ../cache/baseline/debthunter/unique/input/unique_input.csv
I'm using your pre-trained models.
Done!
I labeled the comments! Now I save them!
Wrote 6531 predictions to ../cache/baseline/debthunter/unique/output/unique_input_predictions.csv
Test Result:
              precision    recall  f1-score   support

          no      0.992     0.982     0.987      6418
         yes      0.339     0.531     0.414       113

    accuracy                          0.974      6531
   macro avg      0.665     0.756     0.700      6531
weighted avg      0.980     0.974     0.977      6531

Running: java -jar ../lib/DebtHunter-0.0.1-SNAPSHOT.jar -u second -l ../cache/baseline/debthunter/train/training.arff -o ../cache/baseline/debthunter/model
Let's start!
You selected the second use case!
The training data is ok!
1st step: Paramenters optimization started.
Grid search with DebtHunter bin

## 5-Fold Cross Validation

Runs 5-fold cross validation (mirroring `detect-liu-detector.ipynb`) for both
the **unique** and **duplicate** datasets. Each fold retrains DebtHunter from
scratch, so this takes a long time.


In [13]:
from sklearn.model_selection import StratifiedGroupKFold

for dataset_name in ['unique', 'duplicate']:
    print(f'\n===== {dataset_name} (5-Fold CV) =====')
    base_df = pd.concat([
        pd.read_csv(f'../data/{dataset_name}_detect_train.csv'),
        pd.read_csv(f'../data/{dataset_name}_detect_test.csv'),
    ])
    X = base_df['text']
    y = base_df['label']
    groups = base_df['repository']

    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups)):
        fold_suffix = f'5fcv-{fold + 1}'
        train_df = base_df.iloc[train_idx]
        test_df = base_df.iloc[test_idx]
        print(f'Fold {fold_suffix}: {len(train_df)} train / {len(test_df)} test')

        fold_detector = DebtHunterSatdDetectorModel(
            'detect',
            f'trained-debthunter-{fold_suffix}',
            simple_output_label_converter,
            BASE_DEBT_HUNTER_DIRECTORY,
            retrain=True,
        )
        fold_detector.fit(Dataset.from_pandas(train_df))
        fold_detector.predict(Dataset.from_pandas(test_df), dataset_name)



===== unique (5-Fold CV) =====
Fold 5fcv-1: 27030 train / 5525 test
Running: java -jar ../lib/DebtHunter-0.0.1-SNAPSHOT.jar -u second -l ../cache/baseline/debthunter/train/training.arff -o ../cache/baseline/debthunter/model
Let's start!
You selected the second use case!
The training data is ok!
1st step: Paramenters optimization started.
Grid search with DebtHunter binary: the optimal gamma and the optimal cost are [2.0, 0.0]
1st step: Training phase started.
2nd step: Paramenters optimization started.
Grid search with DebtHunter multi: the optimal gamma and the optimal cost are [2.0, 2.0]
2nd step: Training phase started.
Models training ended!
trained-debthunter-5fcv-1
Let's start!
You selected the first use case!
Loaded 5525 comment rows from ../cache/baseline/debthunter/unique/input/unique_input.csv
I'm using your pre-trained models.
Done!
I labeled the comments! Now I save them!
Wrote 5525 predictions to ../cache/baseline/debthunter/unique/output/unique_input_predictions.csv
Test